In [ ]:
!git clone -b pandax \
https://github.com/saisua/jax-pandax.git

Cloning into 'jax-pandax'...
remote: Enumerating objects: 184653, done.
remote: Counting objects: 100% (1009/1009), done.
remote: Compressing objects: 100% (546/546), done.
remote: Total 184653 (delta 717), reused 463 (delta 463), pack-reused 183644 (from 4)
Receiving objects: 100% (184653/184653), 115.23 MiB | 21.09 MiB/s, done.
Resolving deltas: 100% (146813/146813), done.


In [ ]:
%pip install --upgrade 'jax<0.6.0' jaxlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 MB 8.2 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.5.1
    Uninstalling jaxlib-0.5.1:
      Successfully uninstalled jaxlib-0.5.1
  Attempting uninstall: jax
    Found existing installation: jax 0.5.2
    Uninstalling jax-0.5.2:
      Successfully uninstalled jax-0.5.2


In [ ]:
!cp -R ./jax-pandax/jax/experimental/pandax ./

In [ ]:
!find ./pandax -type f -name "*.py" -exec sed -i 's/jax\.experimental\.pandax\././g' {} +

In [ ]:
#!cd jax && \
#python build/build.py && \
#pip install dist/*.whl

In [ ]:
import pandax as jpd
import pandas as pd

import plotly.express as px

/usr/local/lib/python3.11/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.1 is installed, but it is not compatible with the installed jaxlib version 0.5.3, so it will not be used.
  warnings.warn(


##CODE

### SQL

### Series

In [ ]:
from __future__ import annotations

from typing import Any, List, Optional, Tuple, Union
from functools import partial

import jax
from jax import tree_util
import jax.numpy as jnp
import numpy as np
import pandas as pd

from pandax.index import Index
from pandax.index import RangeIndex
from pandax.stringarray import asarray_or_stringarray
from pandax.stringarray import StringArray
from pandax.series import SeriesGroupBy

class Series:
  """Series implementation built on JAX."""
  _index: Index
  _data: Union[jnp.ndarray, StringArray]
  _pd_value: Optional[pd.Series] = None

  index = property(lambda self: self._index)
  dtype = property(lambda self: self._data.dtype)
  shape = property(lambda self: self._data.shape)
  size = property(lambda self: self._data.size)
  ndim = property(lambda self: self._data.ndim)

  def __init__(self, data, index=None, dtype=None):
    if isinstance(data, Series):
      if index is None:
        index = data._index
      data = data._data

    data = asarray_or_stringarray(data, dtype=dtype)

    if data.ndim != 1:
      raise ValueError("Series data must be 1-dimensional")

    self._data = data

    if index is None:
      index = RangeIndex(len(self._data))
    elif not isinstance(index, Index):
      index = Index([
        ind
        if not isinstance(ind, StringArray) else
        str(ind)
        for ind in index
      ])

    self._index = index

  @property
  def _value(self) -> pd.Series:
    if self._pd_value is None:
      self._pd_value = pd.Series(
          np.asarray(self._data), index=np.asarray(self._index))
    return self._pd_value

  @property
  def values(self) -> jnp.ndarray:
    return self._data

  def to_pandas(self) -> pd.Series:
    return self._value

  def __array__(self) -> np.ndarray:
    return np.array(self._data)

  def __repr__(self) -> str:
    return repr(self._value)

  def __iter__(self):
    return iter(self.index)

  def __len__(self) -> int:
    return len(self._data)

  def __getitem__(self, ind):
    if isinstance(ind, slice):
      return Series(self._data[ind], self._index[ind])
    ind = asarray_or_stringarray(ind)
    if ind.ndim > 1:
      raise ValueError(f"Too many indices for Series: {ind}")

    indexer = self._index.get_indexer(ind.ravel())
    if ind.ndim == 0:
      return self._data[indexer[0]]
    return Series(self._data[indexer], self._index[indexer])

  def __add__(self, other) -> Series:
    return self._add(self._data, other)

  @staticmethod
  def _add(data, other):
    return Series(data + other)

  # def __sub__

  def groupby(self, by: Any) -> SeriesGroupBy:
    return SeriesGroupBy(self, by)

  def _tree_flatten(self) -> Tuple[List[jnp.ndarray], tree_util.PyTreeDef]:
    return tree_util.tree_flatten([self._index, self._data])

  @classmethod
  def _tree_unflatten(cls, aux_data: tree_util.PyTreeDef,
                      children: List[jnp.ndarray]) -> Series:
    obj = cls([])
    obj._index, obj._data = tree_util.tree_unflatten(aux_data, children)
    return obj

  @property
  def mean(self):
    return partial(self._mean, self._data)

  @staticmethod
  @jax.jit
  def _mean(data: jnp.ndarray) -> float:
    return jnp.mean(data)

  @property
  def median(self):
    return partial(self._median, self._data)

  @staticmethod
  @jax.jit
  def _median(data: jnp.ndarray) -> float:
    return jnp.median(data)

  @property
  def min(self):
    return self.partial(self._min, self._data)

  @staticmethod
  @jax.jit
  def _min(data: jnp.ndarray) -> float:
    return jnp.min(data)

  @property
  def max(self) -> float:
    return partial(self._max, self._data)

  @staticmethod
  @jax.jit
  def _max(data: jnp.ndarray) -> float:
    return jnp.max(data)

  @property
  def std(self):
    return partial(self._std, self._data)

  @staticmethod
  @jax.jit
  def _std(data: jnp.ndarray) -> float:
    return jnp.std(_data)

tree_util.register_pytree_node(
    Series,
    lambda obj: obj._tree_flatten(),  # pylint: disable=protected-access
    Series._tree_unflatten)  # pylint: disable=protected-access

In [ ]:
a = Series([1, 2, 3])

a + 2

0    3
1    4
2    5
dtype: int32

### DF

In [ ]:
from __future__ import annotations

from typing import Any, Dict, List, Optional, Tuple, Union
from functools import partial
import csv
from math import ceil

from jax import tree_util
import jax.numpy as jnp
import numpy as np
import pandas as pd
import jax

from pandax.dataframe import DataFrameGroupBy
from pandax.index import Index, RangeIndex
from pandax.stringarray import StringArray

IndexLike = Union[Any, Index]
SeriesLike = Union[Any, Series, List, Tuple, np.ndarray, jnp.ndarray]
DFLike = Union[pd.DataFrame, "DataFrame"]

class classproperty(property):
    def __get__(self, cls, owner):
        return classmethod(self.fget).__get__(None, owner)()

class DataFrame:
  """DataFrame implementation built on JAX."""
  _column_index: Index
  _row_index: Index
  _columns: Tuple[Union[jnp.ndarray, StringArray]]
  _pd_value: Optional[pd.DataFrame] = None

  shape = property(lambda self: (len(self._row_index), len(self._column_index)))
  size = property(lambda self: len(self._row_index) * len(self._column_index))
  ndim = property(lambda self: 2)
  index = property(lambda self: self._row_index)
  columns = property(lambda self: self._column_index)

  def __init__(self,
               data: Union[Dict[IndexLike, SeriesLike], DFLike, SeriesLike],
               *,
               index: Optional[IndexLike] = None,
               columns: Optional[IndexLike] = None,
               transpose: bool=False,
               ):
    if isinstance(data, dict):
      self._parse_dict_data(data, index, columns)
    elif isinstance(data, (pd.DataFrame, DataFrame)):
      self._parse_pandas_data(data, index, columns)
    elif isinstance(data, (list, tuple, np.ndarray, jnp.ndarray, Series)):
      self._parse_array_data(data, index, columns, transpose)
    else:
      raise NotImplementedError(f"Unsupported data type \"{type(data)}\" for DataFrame initialization.")

  def _parse_dict_data(self, data: Dict[Any, SeriesLike], index: Optional[IndexLike], columns: Optional[IndexLike]):
    """Parse data from a dictionary."""
    if columns is None:
      columns = data.keys()
    if not isinstance(columns, Index):
      columns = Index([
        col
        if not isinstance(col, StringArray) else
        str(col)
        for col in columns
      ])

    self._column_index = columns

    if index is None:
      index = data[0].index if data else RangeIndex(0)
    if not isinstance(index, Index):
      index = Index([
        ind
        if not isinstance(ind, StringArray) else
        str(ind)
        for ind in index
      ])

    self._row_index = Index(index)

    data = tuple(Series(val, index=index) for val in data.values())

    assert all(col.ndim == 1 for col in data)
    assert len({len(col) for col in data}) < 2, f"All columns must have the same length { {len(col) for col in data} }"

    if data:
      assert len(index) == len(data[0]), f"Index length must match column length ({len(index)} == {len(data[0])})"

    self._columns = tuple(col._data for col in data)

  def _parse_pandas_data(self, data: DFLike, index: Optional[IndexLike], columns: Optional[IndexLike]):
    """Parse data from a pandas DataFrame."""
    # Use pandas DataFrame index and columns
    if columns is None:
      columns = data.columns
    if not isinstance(columns, Index):
      columns = Index([
        col
        if not isinstance(col, StringArray) else
        str(col)
        for col in columns
      ])

    self._column_index = columns

    if index is None:
      index = data.index
    if not isinstance(index, Index):
      index = Index([
        ind
        if not isinstance(ind, StringArray) else
        str(ind)
        for ind in index
      ])

    self._row_index = index

    # Convert each column to Series-like JAX arrays or StringArray, if needed
    self._columns = tuple(jnp.array(data[col].values) for col in data.columns)


  def _parse_array_data(self,
                        data: SeriesLike,
                        index: Optional[IndexLike],
                        columns: Optional[IndexLike],
                        transpose: bool=False,
                        ):
    """Parse data from a list, numpy array, or jax array."""

    # Convert list to a numpy array for easier handling of dimensions
    if isinstance(data, Series):
        data = data._data
    elif isinstance(data, (list, tuple)):
      new_data = []
      for d in data:
        if isinstance(d, Series):
            new_data.append(d._data)
        else:
            new_data.append(d)

      data = jnp.array(new_data)
      del new_data

    if transpose:
        data = data.T

    # Ensure data is 1D or 2D
    if data.ndim == 1:
      data = data.reshape(-1, 1)  # Convert 1D array to 2D column vector
    elif data.ndim != 2:
      raise ValueError(f"Only 1D or 2D arrays are supported (got {data.ndim}).")

    num_rows, num_columns = data.shape

    # Default index if not provided
    if index is None:
      index = RangeIndex(num_rows)
    else:
      if len(index) != num_rows:
        raise ValueError(f"Length of index does not match number of rows ({len(index)} vs {num_rows}).")

      if not isinstance(index, Index):
        index = Index([
          ind
          if not isinstance(ind, StringArray) else
          str(ind)
          for ind in index
        ])

    self._row_index = index

    # Default columns if not provided
    if columns is None:
      columns = RangeIndex(num_columns)
    else:
      if len(columns) != num_columns:
        raise ValueError(f"Length of columns does not match number of data columns ({len(columns)} vs {(new_columns)}).")

      if not isinstance(columns, Index):
        columns = Index([
          col
          if not isinstance(col, StringArray) else
          str(col)
          for col in columns
        ])

    self._column_index = columns

    # Assign the parsed data to internal structure
    self._columns = tuple(jnp.array(data[:, i]) for i in range(num_columns))

  @property
  def _value(self) -> pd.DataFrame:
    if self._pd_value is None:
      self._pd_value = pd.DataFrame(
          dict(zip(self._column_index, self._columns)), self._row_index._data)  # pylint: disable=protected-access
    return self._pd_value

  def to_pandas(self) -> pd.DataFrame:
    return self._value

  def to_jax_numpy(self) -> jnp.ndarray:
    return jnp.stack(self._columns)

  def to_numpy(self) -> np.ndarray:
    return np.array(self.to_jax_numpy())

  def __repr__(self) -> str:
    return repr(self._value)

  def __len__(self) -> int:
    return len(self._row_index)

  def __iter__(self):
    return iter(self.columns)

  def __getattr__(self, name):
    """Enable attribute-like access to columns."""
    if name in self._column_index:
      # Find the index of the column with this name
      return self.__getitem__(name)
    else:
      # If the attribute isn't a column, raise an AttributeError
      raise AttributeError(f"'{self.__class__.__name__}' object has no attribute '{name}'")

  def __getitem__(self, item) -> Union[Series, 'DataFrame']:
    # Column selection by name (e.g., `df['column_name']`)
    if isinstance(item, str):
      idx, = self._column_index.get_indexer([item])
      return Series(self._columns[idx], index=self._row_index)

    # Row selection by integer/slice (e.g., `df[3]` or `df[2:5]`)
    elif isinstance(item, int):
      # Selecting a single row
      row_data = [col[item] for col in self._columns]
      return Series(row_data, index=self._column_index)

    elif isinstance(item, slice):
      # Selecting a range of rows
      sliced_data = [col[item] for col in self._columns]

      dict_data = dict()
      for i in range(len(self._columns)):
        key = self._column_index[i]
        if isinstance(key, jax.Array):
            key = int(key)
        dict_data[key] = sliced_data[i]

      return DataFrame(
        dict_data,
        index=self._row_index[item]
      )

    # Column selection by list of names (e.g., `df[['col1', 'col2']]`)
    elif isinstance(item, list) and all(isinstance(i, str) for i in item):
      # Check that each requested column name exists in the column index
      col_indices = self._column_index.get_indexer(item)
      selected_columns = [self._columns[i] for i in col_indices]
      return DataFrame(
        {name: self._columns[i] for name, i in zip(item, col_indices)},
        index=self._row_index
      )

    # Row and Column selection by tuple (e.g., `df[2:5, ['col1', 'col2']]`)
    elif isinstance(item, tuple) and len(item) == 2:
      row_sel, col_sel = item

      # Row selection (either integer, slice, or index list)
      if isinstance(row_sel, int):
        selected_rows = [col[row_sel] for col in self._columns]
        row_index = self._row_index[row_sel]
        row_data = Series(selected_rows, index=self._column_index)
        if isinstance(col_sel, str):  # Single column from the selected row
          col_idx, = self._column_index.get_indexer([col_sel])
          return row_data[col_idx]
        elif isinstance(col_sel, list):  # Specific columns from the selected row
          col_indices = self._column_index.get_indexer(col_sel)
          return Series([selected_rows[i] for i in col_indices], index=col_sel)
        return row_data  # Return full row data if no column selection

      elif isinstance(row_sel, slice):
        row_index = self._row_index[row_sel]
        row_data = {self._column_index[i]: col[row_sel] for i, col in enumerate(self._columns)}

      # Column selection within selected rows
      if isinstance(col_sel, str):  # Single column
        col_idx, = self._column_index.get_indexer([col_sel])
        return Series(self._columns[col_idx][row_sel], index=row_index)
      elif isinstance(col_sel, list):  # Multiple columns
        col_indices = self._column_index.get_indexer(col_sel)
        selected_data = {self._column_index[i]: self._columns[i][row_sel] for i in col_indices}
        return DataFrame(selected_data, index=row_index)

    raise NotImplementedError(f"getitem {item}")

  def __add__(self, other):
    assert set(self._column_index) == set(other._column_index)

    result = self._add(self.to_jax_numpy(), other.to_jax_numpy())

    result._column_index = StringArray(self._column_index)
    return result

  @staticmethod
  @jax.jit
  def _add(data, other_data):
      return DataFrame(
        data + other_data,
        transpose=True
     )

  @property
  def at(self):
    return self.to_jax_numpy().at

  @property
  def T(self):
    return DataFrame(
      self.to_jax_numpy(),
      index=self.columns,
      columns=self.index,
   )

  def _tree_flatten(self) -> Tuple[List[jnp.ndarray], tree_util.PyTreeDef]:
    return tree_util.tree_flatten(
        [self._column_index, self._row_index, self._columns])

  def drop(self, labels: list, axis=0):
    if axis == 0:
      raise NotImplementedError("DataFrame.drop() along axis=0")
    if np.ndim(labels) > 0:
      raise NotImplementedError("DataFrame.drop() with multiple labels")

    if labels not in self.columns:
      raise KeyError(f"{labels} not found in {self.columns}")

    data = dict((key, col)
                for key, col in zip(self.columns, self._columns)
                if key != labels)
    return self.__class__(data, index=self.index)

  @classmethod
  def _tree_unflatten(cls, aux_data: tree_util.PyTreeDef,
                      children: List[jnp.ndarray]) -> DataFrame:
    obj = object.__new__(cls)
    obj._column_index, obj._row_index, obj._columns = tree_util.tree_unflatten(
        aux_data, children)
    return obj

  def groupby(self, by: Any) -> DataFrameGroupBy:
    return DataFrameGroupBy(self, by)

  def head(self, n: int = 5) -> 'DataFrame':
    """Return the first `n` rows of the DataFrame as a new CustomDataFrame."""
    return self[:n]

  def tail(self, n: int = 5) -> 'DataFrame':
    """Return the last `n` rows of the DataFrame as a new CustomDataFrame."""
    return self[-n:]

  @property
  def corr(self):
    return partial(self._corr, self.to_jax_numpy(), self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=['columns'])
  def _corr(data: jnp.ndarray, columns: list) -> "DataFrame":
    return DataFrame(
      jnp.corrcoef(data),
      index=columns,
      columns=columns,
    )

  @property
  def mean(self):
    return partial(self._mean, self._columns, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=['columns'])
  def _mean(data: Tuple[jnp.ndarray, ...], columns: list) -> Series:
    return Series(
      list(map(jnp.mean, data)),
      index=columns
   )

  @property
  def median(self):
    return partial(self._median, self._columns, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=['columns'])
  def _median(data: Tuple[jnp.ndarray, ...], columns: list) -> Series:
    return Series(
      list(map(jnp.median, data)),
      index=columns
   )

  @property
  def min(self):
    return partial(self._min, self._columns, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=['columns'])
  def _min(data: Tuple[jnp.ndarray, ...], columns: list) -> Series:
    return Series(
      list(map(jnp.min, data)),
      index=columns
   )

  @property
  def max(self):
    return partial(self._max, self._columns, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=['columns'])
  def _max(data: Tuple[jnp.ndarray, ...], columns: list) -> Series:
    return Series(
      list(map(jnp.max, data)),
      index=columns
   )

  @property
  def std(self):
    return partial(self._std, self._columns, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=['columns'])
  def _std(data: Tuple[jnp.ndarray, ...], columns: list) -> Series:
    return Series(
      list(map(jnp.std, data)),
      index=columns
   )

  @property
  def quantile(self):
    return partial(self._quantile, self._columns, self.index, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=['index', 'columns', 'q', 'axis', 'numeric_only', 'interpolation'])
  def _quantile(
               data: Tuple[jnp.ndarray, ...],
               index: list,
               columns: list,
               q: Union[float, List[float]],
               axis: Union[Literal['index'], Literal['columns'], Literal[0], Literal[1]]=0,
               numeric_only: bool = True,
               interpolation: str = 'linear'
               ) -> Union["DataFrame", Series]:
    """
    Calculate quantiles for each column in the DataFrame.

    Args:
        q: float or list of floats in [0, 1], the quantile(s) to compute.
        axis: Axis along which to compute (0 for columns, 1 for rows).
        numeric_only: If True, include only numeric data.
        interpolation: Interpolation method (only 'linear' supported with jnp.quantile).

    Returns:
        DataFrame or Series with quantile values.
    """
    # Validate `q`
    if isinstance(q, float):
       q = [q]
    if not isinstance(q, jnp.ndarray):
       q = jnp.array(q)

    if axis == 0 or axis == 'index':
        # Calculate quantiles for each column
        quantiles = []
        for i, col in enumerate(data):
            if numeric_only and not jnp.issubdtype(col.dtype, jnp.number):
                continue
            quantiles.append(jnp.quantile(col, q, method=interpolation))

        if  len(q) == 1:
            # Convert results to DataFrame
            return Series(
                jnp.concatenate(quantiles),
                index=columns
            )
        else:
            return DataFrame(
                jnp.stack(quantiles),
                columns=columns,
                index=q,
            )

    elif axis == 1 or axis == 'columns':
        # Calculate quantiles for each row
        row_quantiles = []
        for row_idx in range(data[0].shape[0]):
            row_values = jnp.array([col[row_idx] for col in data if (not numeric_only or jnp.issubdtype(col.dtype, jnp.number))])
            row_quantiles.append(jnp.quantile(row_values, q, method=interpolation))

        # Stack into DataFrame with appropriate index
        if len(q) == 1:
            return Series(
                jnp.concatenate(row_quantiles),
                index=index,
            )
        else:
            return DataFrame(
                jnp.stack(row_quantiles),
                columns=q,
                index=index,
            )

    else:
        raise ValueError("Axis must be 0 or 1.")

  @property
  def describe(self):
    return partial(self._describe, self._columns, self.index, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=["index", "columns"])
  def _describe(data: Tuple[jnp.ndarray, ...], index: list, columns: list) -> "DataFrame":
    return DataFrame(
      [
        DataFrame._min(data, columns),
        DataFrame._quantile(data, index, columns, 0.25),
        DataFrame._mean(data, columns),
        DataFrame._median(data, columns),
        DataFrame._quantile(data, index, columns, 0.75),
        DataFrame._max(data, columns),
        DataFrame._std(data, columns),
      ],
      columns=columns,
      index=[
        "min",
        "25%",
        "mean",
        "median",
        "75%",
        "max",
        "std",
      ]
    )

  @property
  def fillna(self):
    return partial(self._fillna, self._columns, self.index, self.columns)

  @staticmethod
  @partial(jax.jit, static_argnames=["index", "columns"])
  def _fillna(data: Tuple[jnp.ndarray, ...], index: list, columns: list, value: Union[jnp.number, jnp.ndarray]=0) -> "DataFrame":
    return DataFrame(
        [
            jnp.where(jnp.isnan(data), value, data)
            for data in data
        ],
        index=index,
        columns=columns,
        transpose=True,
    )

  @property
  def dropna(self):
    return partial(
        self._dropna,
        self._columns,
        self.index,
        self.columns,
    )

  @staticmethod
  # @partial(jax.jit, static_argnames=['index', 'columns', 'axis', 'how', 'subset', 'ignore_index'])
  def _dropna(
             data: Tuple[jnp.ndarray, ...],
             index: list,
             columns: list,
             axis: Union[Literal['index'], Literal['columns'], Literal[0], Literal[1]]=0,
             how: Union[Literal['all'], Literal['any']]='any',
             thresh: Optional[float]=None,
             subset: Optional[Union[str, List[str]]]=None,
             ignore_index: bool=False
             ) -> "DataFrame":
    """Drop rows or columns with NaN values based on conditions.

    Parameters:
    axis : int, default 0
        If 0, drop rows with NaN values.
        If 1, drop columns with NaN values.
    how : {'any', 'all'}, default 'any'
        Determine if row or column is removed when encountering NaN.
        - 'any' : If any NaN values are present, drop that row or column.
        - 'all' : If all values are NaN, drop that row or column.
    thresh : int, optional
        Require that many non-NaN values to keep a row/column.
    subset : list, optional
        Specify columns (if axis=0) or rows (if axis=1) to check for NaN values.
    ignore_index : bool, default False
        If True, reset index in the result.

    Returns:
    DataFrame : New DataFrame with NaN values dropped based on the specified criteria.
    """

    if subset is None:
        # Use all columns for checking NaNs if subset is not provided
        columns_to_check = data
        col_idx = range(len(data))
    else:
        # Restrict to columns defined in subset
        col_idx = [columns.index(col) for col in subset]
        columns_to_check = tuple(data[i] for i in col_idx)

    data_stack = jnp.stack(columns_to_check, axis=1)  # Stack subset columns for easier NaN checks

    if axis == 0 or axis == 'index':  # Drop rows
        if thresh is not None:
            mask = jnp.sum(~jnp.isnan(data_stack), axis=1) >= thresh
        elif how == 'all':
            mask = ~jnp.all(jnp.isnan(data_stack), axis=1)
        elif how == 'any':
            mask = ~jnp.any(jnp.isnan(data_stack), axis=1)
        else:
            raise ValueError("how must be either 'any' or 'all'")

        new_columns = tuple(col[mask] for col in data)
        new_row_index = [index[i] for i, m in enumerate(mask) if m]
        if ignore_index:
            new_row_index = list(range(len(new_row_index)))

        return DataFrame(new_columns, columns=columns, index=new_row_index, transpose=True)

    elif axis == 1 or axis == 'columns':  # Drop columns
        if thresh is not None:
            mask = jnp.sum(~jnp.isnan(data_stack), axis=0) >= thresh
        elif how == 'all':
            mask = ~jnp.all(jnp.isnan(data_stack), axis=0)
        elif how == 'any':
            mask = ~jnp.any(jnp.isnan(data_stack), axis=0)
        else:
            raise ValueError("how must be either 'any' or 'all'")

        new_columns = tuple(col for i, col in enumerate(data) if i in col_idx and mask[i])
        new_column_index = [columns[i] for i, m in enumerate(mask) if m]

        return DataFrame(new_columns, index=index, columns=new_column_index, transpose=True)

    else:
        raise ValueError("axis must be 0 or 1")

  @classproperty
  def read_csv(cls):
    return DataFrame._read_csv

  @staticmethod
  # @partial(jax.jit, static_argnames=['delimiter'])
  def _read_csv(file: str, delimiter: str=' ', shape: tuple[int, int]=None, columns: list[str]=None, index: str | list[str]=None) -> "DataFrame":
    if shape is None:
      csv_shape = jax.experimental.io_callback(
        DataFrame._read_csv_shape_callback(
            file,
            delimiter=delimiter
        ),
        jax.ShapeDtypeStruct(
            (2,),
            jnp.int32
        ),
      )
    else:
      csv_shape = shape
    data = jax.experimental.io_callback(
        DataFrame._read_csv_callback(
            file,
            delimiter=delimiter
        ),
        jax.ShapeDtypeStruct(
            csv_shape,
            jnp.float32
        ),
    )

    if columns is not None:
      if index is None:
        index = "index"
      if isinstance(index, str) and index in columns:
        index_pos = columns.index(index)
        columns.pop(index_pos)
        return DataFrame(
            jnp.concatenate((
                    data[:, :index_pos],
                    data[:, index_pos+1:],
                ),
                axis=1,
            ),
            columns=StringArray(columns),
            index=data[index_pos]
        )
      elif isinstance(index, int):
        columns.pop(index)
        return DataFrame(
            jnp.concatenate((
                    data[:, :index],
                    data[:, index+1:],
                ),
                axis=1,
            ),
            columns=StringArray(columns),
            index=data[index]
        )
      elif isinstance(index, (list, tuple, jax.Array)):
        return DataFrame(
            data,
            index=index,
            columns=StringArray(columns),
        )

    return DataFrame(
      data,
    )

  @staticmethod
  def _read_csv_shape_callback(file: str, delimiter: str):
    def __read_csv_shape_callback() -> jax.Array:
        with open(file, 'r') as csvfile:
            csv_reader = csv.reader(csvfile)
            rows = next(csv_reader)
            return jnp.array((len([0 for _ in csv_reader]), len(rows)))
    return __read_csv_shape_callback

  @staticmethod
  def _read_csv_callback(file: str, delimiter: str):
    def __read_csv_callback() -> jax.Array:  # cols, rows
        with open(file, 'r') as csvfile:
            csv_reader = csv.reader(csvfile)
            next(csv_reader)
            return jnp.array([[float(c) for c in row] for row in csv_reader])
    return __read_csv_callback

  @property
  def to_csv(self):
    return partial(self._to_csv, self._columns, self.index, self.columns)

  @staticmethod
  #@jax.jit
  def _to_csv(
              data: Tuple[jnp.ndarray, ...],
              index: list,
              columns: list,
              file: str,
              delimiter: str=',',
  ):
    jax.experimental.io_callback(
        DataFrame._to_csv_callback(
            index,
            columns,
            file,
            delimiter=delimiter
        ),
        jax.ShapeDtypeStruct(
            tuple(),
            jnp.int32
        ),
        data=data,
    )

  @staticmethod
  def _to_csv_callback(
    index,
    columns,
    file,
    delimiter,
  ):
    def __to_csv_callback(data):
      with open(file, "w+") as csvfile:
            csv_writer = csv.writer(csvfile)
            if False and index is not None:
              csv_writer.writerow(["index", *columns])
              csv_writer.writerows(zip(index, *data))
            else:
              csv_writer.writerow(columns)
              csv_writer.writerows(zip(*data))

      return 0
    return __to_csv_callback

  @classproperty
  def read_sql_table(cls):
    return cls._read_sql_table

  @staticmethod
  @partial(jax.jit, static_argnames=["table_name", "conn", "split_n", "shape"])
  def _read_sql_table(table_name: str, conn: str, split_n=1, shape=None):
    if shape is None:
      data_shape = jax.experimental.io_callback(
        DataFrame._read_sql_table_shape_callback(
            table_name,
            conn,
        ),
        jax.ShapeDtypeStruct(
            (2,),
            jnp.int32
        ),
      )
    else:
      data_shape = shape
    if split_n == 1:
        data = jax.experimental.io_callback(
            DataFrame._read_sql_table_callback(
                table_name,
                conn,
            ).__next__,
            jax.ShapeDtypeStruct(
                data_shape,
                jnp.float32
            )
        )
    else:
        num_cols = data_shape[1]
        num_rows = data_shape[0]
        split_size = int(ceil(num_rows / split_n))
        data_list = list()
        callback = DataFrame._read_sql_table_callback(
          table_name,
          conn,
          split_n,
          split_size,
        )
        for split in range(split_n):
            # print(skip)
            data = jax.experimental.io_callback(
                callback.__next__,
                jax.ShapeDtypeStruct(
                    (min(split_size, num_rows - split*split_size), num_cols),
                    jnp.float32
                ),
            )

            data_list.append(data)

        if split_n <= 0:
            data = []
        else:
            data = jnp.concatenate(data_list, axis=1)

    return DataFrame(data)

  @staticmethod
  def _read_sql_table_shape_callback(table_name: str, conn_str: str):
    def __read_sql_table_shape_callback(table_name=table_name, conn_str=conn_str):
        conn = sqlite3.connect(conn_str)
        cursor = conn.cursor()

        # Get number of rows
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        row_count = int(cursor.fetchone()[0])

        # Get number of columns
        cursor.execute(f"PRAGMA table_info({table_name})")
        column_count = len(cursor.fetchall())

        shape = jnp.array([row_count, column_count])  # cols, rows
        conn.close()
        return shape
    return __read_sql_table_shape_callback

  @staticmethod
  def _read_sql_table_callback(table_name: str, conn_str: str, n_splits: int=1, split_size=None):
    while True:
        conn = sqlite3.connect(conn_str)
        cursor = conn.cursor()

        query = [
            f"SELECT * FROM {table_name}"
        ]

        cursor.execute(' '.join(query))
        for _ in range(n_splits - 1):
            yield jnp.array(cursor.fetchmany(split_size), dtype=jnp.float32)

        #print(repr(data))
        data = jnp.array(cursor.fetchall(), dtype=jnp.float32)
        conn.close()
        yield data
        del data

  @staticmethod
  def _read_sql_table_callbackBU(table_name: str, conn: str):
    def __read_sql_table_callback(skip: int=0, limit: int=None, *, table_name=table_name, conn=conn):
        conn = sqlite3.connect(conn)
        cursor = conn.cursor()

        query = [
            f"SELECT * FROM {table_name}"
        ]
        if limit is not None and limit:
            query.append(f"LIMIT {int(limit)}")
        if skip:
            query.append(f"OFFSET {int(skip)}")

        cursor.execute(' '.join(query))
        data = cursor.fetchall()
        #print(repr(data))
        return jnp.array(data, dtype=jnp.float32)
    return __read_sql_table_callback

  @staticmethod
  def _read_sql_tableBU(table_name: str, conn: str, split_n=1, shape=None):
    if shape is None:
      data_shape = jax.experimental.io_callback(
        DataFrame._read_sql_table_shape_callback(
            table_name,
            conn,
        ),
        jax.ShapeDtypeStruct(
            (2,),
            jnp.int32
        ),
      )
    else:
      data_shape = shape
    if split_n == 1:
        data = jax.experimental.io_callback(
            DataFrame._read_sql_table_callback(
                table_name,
                conn,
            ),
            jax.ShapeDtypeStruct(
                data_shape,
                jnp.float32
            )
        )
    else:
        num_cols = int(data_shape[1])
        num_rows = int(data_shape[0])
        split_size = int(jnp.ceil(num_rows / split_n))
        data_list = list()
        for skip in range(0, int(num_rows), split_size):
            # print(skip)
            data = jax.experimental.io_callback(
                DataFrame._read_sql_table_callback(
                    table_name,
                    conn,
                ),
                jax.ShapeDtypeStruct(
                    (min(split_size, num_rows - skip), num_cols),
                    jnp.float32
                ),
                skip=skip,
                limit=split_size
            )

            data_list.append(data)
        if len(data_list) == 0:
            data = []
        elif len(data_list) == 1:
            data = data_list[0]
        else:
            data = jnp.concatenate(data_list, axis=1)

    return DataFrame(data)

tree_util.register_pytree_node(
    DataFrame,
    lambda obj: obj._tree_flatten(),  # pylint: disable=protected-access
    DataFrame._tree_unflatten
)  # pylint: disable=protected-access


# jdf = DataFrame.read_sql_table("table_name", "database.db", split_n=2)

# Tests

In [ ]:
csv_path = '/content/sample_data/california_housing_train.csv'

In [ ]:
import sqlite3
import pandas as pd

# Read CSV file into pandas DataFrame
df = pd.read_csv(csv_path)

# Connect to SQLite database (creates if doesn't exist)
conn = sqlite3.connect('database.db')

# Write DataFrame to SQLite table
df.to_sql('table_name', conn, if_exists='replace', index=False)

# Close connection
conn.close()

In [ ]:
#jdf = DataFrame.read_csv(csv_path)
jdf = DataFrame.read_sql_table("table_name", "database.db", split_n=5, shape=(17000, 9))

In [ ]:
#jdf = DataFrame(df.to_dict('list'))
df = pd.read_csv(csv_path)
jdf2 = DataFrame(df)
jdf2.to_csv("test.csv")
jdf = DataFrame.read_csv(
  "test.csv",
  columns=jdf.columns
)

In [ ]:
jdf.head(3).to_pandas()

,0,1,2,3,4,5,6,7,8
0,-114.309998,34.189999,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.470001,34.400002,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.559998,33.689999,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0


In [ ]:
jdf2.head(3).to_pandas()

NameError: name 'jdf2' is not defined

In [ ]:
jdf.shape

(17000, 9)

In [ ]:
(jdf + jdf).head(3).to_pandas()

,0,1,2,3,4,5,6,7,8
0,-228.619995,68.379997,30.0,11224.0,2566.0,2030.0,944.0,2.9872,133800.0
1,-228.940002,68.800003,38.0,15300.0,3802.0,2258.0,926.0,3.6400,160200.0
2,-229.119995,67.379997,34.0,1440.0,348.0,666.0,234.0,3.3018,171400.0


In [ ]:
nandf = df.copy()
nandf.loc[2, "housing_median_age"] = np.nan
nanjdf = DataFrame(nandf)

In [ ]:
nanjdf.fillna(-3).head().to_pandas()

NameError: name 'nanjdf' is not defined

In [ ]:
nanjdf.dropna().head().to_pandas()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.309998,34.189999,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.470001,34.400002,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
3,-114.570000,33.639999,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.570000,33.570000,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0
5,-114.580002,33.630001,29.0,1387.0,236.0,671.0,239.0,3.3438,74000.0


In [ ]:
nanjdf.dropna(axis='columns').head().to_pandas()

,longitude,latitude,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.309998,34.189999,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.470001,34.400002,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.559998,33.689999,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.570000,33.639999,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.570000,33.570000,1454.0,326.0,624.0,262.0,1.9250,65500.0


In [ ]:
jdf.describe().T.to_pandas()

,min,25%,mean,median,75%,max,std
0,-124.349998,-121.790001,-119.562096,-118.489998,-118.000000,-114.309998,2.005107
1,32.540001,33.930000,35.625225,34.250000,37.720001,41.950001,2.137277
2,1.000000,18.000000,28.589354,29.000000,37.000000,52.000000,12.586566
3,2.000000,1462.000000,2643.664551,2127.000000,3151.250000,37937.000000,2179.882812
4,1.000000,297.000000,539.410828,434.000000,648.250000,6445.000000,421.487030
5,3.000000,790.000000,1429.573975,1167.000000,1721.000000,35682.000000,1147.819092
6,1.000000,282.000000,501.221954,409.000000,605.250000,6082.000000,384.509552
7,0.499900,2.566375,3.883578,3.544600,4.767000,15.000100,1.908100
8,14999.000000,119400.000000,207300.921875,180400.000000,265000.000000,500001.000000,115980.351562


In [ ]:
jdf.total_rooms.mean()

AttributeError: 'DataFrame' object has no attribute 'total_rooms'

In [ ]:
jdf.longitude

AttributeError: 'DataFrame' object has no attribute 'longitude'

In [ ]:
jdf[0]

0     -114.309998
1       34.189999
2       15.000000
3     5612.000000
4     1283.000000
5     1015.000000
6      472.000000
7        1.493600
8    66900.000000
dtype: float32

In [ ]:
jdf.tail()

                0          1     2       3      4       5      6       7  \
16995 -124.260002  40.580002  52.0  2217.0  394.0   907.0  369.0  2.3571   
16996 -124.269997  40.689999  36.0  2349.0  528.0  1194.0  465.0  2.5179   
16997 -124.300003  41.840000  17.0  2677.0  531.0  1244.0  456.0  3.0313   
16998 -124.300003  41.799999  19.0  2672.0  552.0  1298.0  478.0  1.9797   
16999 -124.349998  40.540001  52.0  1820.0  300.0   806.0  270.0  3.0147   

              8  
16995  111400.0  
16996   79000.0  
16997  103600.0  
16998   85800.0  
16999   94600.0  

In [ ]:
px.imshow(jdf.corr().to_pandas())

In [ ]:
@jax.jit
def jit_corr_mean(df: DataFrame):
  return df.corr().mean()

jit_corr_mean(jdf)

0    0.020021
1   -0.047587
2   -0.043004
3    0.409024
4    0.389419
5    0.368513
6    0.395537
7    0.185449
8    0.202076
dtype: float32